In [1]:
import sys
from pathlib import Path

# The notebook runs from notebooks/, so climb one level to the project root
# and add it to sys.path so "src.support..." becomes importable.
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.support.EDA_utils import load_all, eda_report, summary_stats

RAW_DIR = PROJECT_ROOT / "data" / "raw"

In [6]:
df_adzuna = load_all(RAW_DIR)
print(df_adzuna.shape)   
df_adzuna.head()


(951, 18)


,id,title,company,category_label,category_tag,location_name,salary_min,salary_max,salary_is_predicted,latitude,longitude,contract_type,contract_time,created,description,redirect_url,search_category,source_page
0,5768359579,"Senior Solution Consultant, UKG (WFD & WFC)",Strada,Trabajos en consultoría,consultancy-jobs,"Madrid, Comunidad de Madrid",NaN,NaN,0,40.416700,3.703250,None,None,2026-06-18T22:22:10Z,"Senior Solution Consultant, UKG (WFD & WFC) St...",https://www.adzuna.es/details/5768359579?utm_m...,consultancy-jobs,p1
1,5784936570,Senior Product Manager - Money (Barcelona),Heetch,Trabajos en consultoría,consultancy-jobs,"Barcelona, Cataluña",65000.0,85000.0,0,41.385100,2.173400,None,None,2026-07-02T06:46:55Z,"Executive Summary Own the vision, strategy and...",https://www.adzuna.es/details/5784936570?utm_m...,consultancy-jobs,p1
2,5781764254,"Director, Viral Vector BD — Gene Therapy Partn...",SK pharmteco,Trabajos en consultoría,consultancy-jobs,"Boiro, A Coruña",70000.0,90000.0,0,42.647057,-8.882718,None,None,2026-06-30T06:43:25Z,SK pharmteco busca un Director de Desarrollo d...,https://www.adzuna.es/details/5781764254?utm_m...,consultancy-jobs,p1
3,5780809431,Director of Assisted Channels Transformation,BANCO SANTANDER,Trabajos en consultoría,consultancy-jobs,"Madrid, Comunidad de Madrid",70000.0,90000.0,0,40.416700,3.703250,None,None,2026-06-29T06:41:35Z,BANCO SANTANDER is seeking a Project Director ...,https://www.adzuna.es/details/5780809431?utm_m...,consultancy-jobs,p1
4,5781765668,Global Expansion Director: Market Strategy,D&M asesores consultores,Trabajos en consultoría,consultancy-jobs,"Madrid, Comunidad de Madrid",60000.0,80000.0,0,40.416700,3.703250,None,None,2026-06-30T06:44:37Z,D&M asesores consultores en Madrid busca un pr...,https://www.adzuna.es/details/5781765668?utm_m...,consultancy-jobs,p1


In [7]:
eda_report(df_adzuna)

EDA REPORT

DATASET SHAPE
Rows: 951, Columns: 18

DATA TYPES
id                      object
title                   object
company                 object
category_label          object
category_tag            object
location_name           object
salary_min             float64
salary_max             float64
salary_is_predicted     object
latitude               float64
longitude              float64
contract_type           object
contract_time           object
created                 object
description             object
redirect_url            object
search_category         object
source_page             object
dtype: object

MISSING VALUES (NaN = field absent in Adzuna response)
contract_type    951
contract_time    924
salary_max       395
salary_min       395
latitude         161
longitude        161
company           91
dtype: int64

Missing as % of rows:
contract_type    100.00
contract_time     97.16
salary_max        41.54
salary_min        41.54
latitude          16.93
longitud

### Salary presence per category

In [8]:
# Salary presence per category: % of postings with a salary value.
# Tests whether the 58.46% global salary presence is uniform across
# categories or hides differences between them.
by_cat = df_adzuna.groupby("search_category").agg(
    total=("id", "size"),
    with_salary=("salary_min", lambda s: s.notna().sum()),
)
by_cat["pct_with_salary"] = (by_cat["with_salary"] / by_cat["total"] * 100).round(1)
print(by_cat)

                  total  with_salary  pct_with_salary
search_category                                      
consultancy-jobs    250          176             70.4
hr-jobs             250          108             43.2
it-jobs             250          131             52.4
legal-jobs          201          141             70.1


In [10]:
# Look at the actual salary_min values: are they round and repeated
# (sign of Adzuna-derived estimates) or dispersed (sign of employer-stated)?
postings_with_salary = df_adzuna[df_adzuna["salary_min"].notna()].copy()

print("Postings with salary:", len(postings_with_salary))
print("Distinct salary_min values:", postings_with_salary["salary_min"].nunique())
print()
print("Most common salary_min values:")
print(postings_with_salary["salary_min"].value_counts().head(15))

Postings with salary: 556
Distinct salary_min values: 62

Most common salary_min values:
salary_min
70000.0     111
60000.0     103
80000.0      72
100000.0     65
90000.0      36
120000.0     32
65000.0      16
40000.0      16
50000.0      11
30000.0       8
35000.0       7
25000.0       7
45000.0       6
27000.0       5
24000.0       4
Name: count, dtype: int64
